<a href="https://colab.research.google.com/github/charlien12/ML-Projects/blob/main/DateFruit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
df=pd.read_csv('DateFruit_Dataset.csv')

In [2]:
df.head()

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [4]:
display(df.isnull().sum())
display(df.shape)

,0
AREA,0
PERIMETER,0
MAJOR_AXIS,0
MINOR_AXIS,0
ECCENTRICITY,0
EQDIASQ,0
SOLIDITY,0
CONVEX_AREA,0
EXTENT,0
ASPECT_RATIO,0


(898, 35)

In [5]:
X=df.drop('Class',axis=1)
Y=df['Class']

In [6]:
from sklearn.preprocessing import StandardScaler,LabelEncoder
le=LabelEncoder()
Y=le.fit_transform(Y)

In [7]:
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=42)

In [8]:
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

# ANN Model And Evaluation

In [9]:
import torch.nn as nn
import torch

In [11]:
X_train_tensor=torch.tensor(X_train_scaled,dtype=torch.float32)
Y_train_tensor=torch.tensor(Y_train,dtype=torch.long)
X_test_tensor=torch.tensor(X_test_scaled,dtype=torch.float32)
Y_test_tensor=torch.tensor(Y_test,dtype=torch.long)

In [12]:
from torch.utils.data import TensorDataset,DataLoader

In [13]:
train_dataset=TensorDataset(X_train_tensor,Y_train_tensor)
test_dataset=TensorDataset(X_test_tensor,Y_test_tensor)

In [14]:
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=32,shuffle=False)

In [15]:
class ANN(nn.Module):
  def __init__(self):
    super(ANN, self).__init__()
    self.model=nn.Sequential(
      nn.Linear(X_train.shape[1],64),
      nn.ReLU(),
      nn.Linear(64,64),
      nn.ReLU(),
      nn.Linear(64,7))
  def forward(self,x):
    return self.model(x)

In [16]:
import torch.optim as optim
model=ANN()
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

# Training the NN

In [17]:
epochs=100
for epoch in range(epochs):
  model.train()
  running_loss=0.0
  for xb,yb in train_loader:
    outputs=model(xb)
    loss=criterion(outputs,yb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    running_loss+=loss.item()
  train_loss=running_loss/len(train_loader)
  print(f"epoch {epoch+1}/{epochs} ==> train loss = {train_loss}")

epoch 1/100 ==> train loss = 1.7371463464654011
epoch 2/100 ==> train loss = 1.1322438716888428
epoch 3/100 ==> train loss = 0.7201356447261312
epoch 4/100 ==> train loss = 0.5416882660077966
epoch 5/100 ==> train loss = 0.4445368990949962
epoch 6/100 ==> train loss = 0.3914415480002113
epoch 7/100 ==> train loss = 0.3504020085801249
epoch 8/100 ==> train loss = 0.3144878965357076
epoch 9/100 ==> train loss = 0.29334314847769943
epoch 10/100 ==> train loss = 0.2697457561026449
epoch 11/100 ==> train loss = 0.2553390860557556
epoch 12/100 ==> train loss = 0.2398352914530298
epoch 13/100 ==> train loss = 0.2246024498473043
epoch 14/100 ==> train loss = 0.21251586209172788
epoch 15/100 ==> train loss = 0.20700368997843369
epoch 16/100 ==> train loss = 0.20358210154201672
epoch 17/100 ==> train loss = 0.19322454151899918
epoch 18/100 ==> train loss = 0.1797564781230429
epoch 19/100 ==> train loss = 0.1821123225533444
epoch 20/100 ==> train loss = 0.1704150551687116
epoch 21/100 ==> train l

# Evaluation

In [19]:
model.eval()
total=0
correct=0
with torch.no_grad():
  for xb,yb in test_loader:
    outputs=model(xb)
    _,predicted=torch.max(outputs,1)
    total+=yb.size(0)
    correct+=(predicted==yb).sum().item()
print(f"Accuracy on test set: {100*correct/total}%")
print(f"Number of correct predictions: {correct}")
print(f"Number of total predictions: {total}")

Accuracy on test set: 95.0%
Number of correct predictions: 171
Number of total predictions: 180
